In [0]:
%scala

// ==========================================
// PROJECT NEXUS
// INCREMENTAL BATCH LOADING
// STEP 1: CONFIGURATION
// ==========================================

val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/"

val incrementalFilePath =
  basePath + "incremental/orders_update.csv"

val bronzeOrdersPath =
  basePath + "bronze/orders"

println("PROJECT NEXUS - Incremental Batch Pipeline")
println("------------------------------------------")
println(s"Incoming File: $incrementalFilePath")
println(s"Bronze Path:   $bronzeOrdersPath")
println("Configuration initialized successfully")


PROJECT NEXUS - Incremental Batch Pipeline
------------------------------------------
Incoming File: abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/incremental/orders_update.csv
Bronze Path: abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders
Configuration initialized successfully
val basePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/
val incrementalFilePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/incremental/orders_update.csv
val bronzeOrdersPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders

In [0]:
%scala
val incrementalFolder =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/incremental/"

display(dbutils.fs.ls(incrementalFolder))

path,name,size,modificationTime
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/incremental/orders_updates.csv,orders_updates.csv,1214,1795252002000


In [0]:
%scala

val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/"

val incrementalPath =
  basePath + "incremental/orders_updates.csv"

val incrementalDF = spark.read
  .option("header", "true")
  .option("inferSchema", "false")
  .option("mode", "PERMISSIVE")
  .csv(incrementalPath)

println(s"Incoming record count: ${incrementalDF.count()}")

display(incrementalDF)


order_id,customer_id,product_id,quantity,order_date,order_status,last_updated
ORD000011,CUST003,PRD022,5,2026-09-26,DELIVERED,2026-09-29 18:58:11
ORD000012,CUST013,PRD017,1,2026-10-28,CANCELLED,2026-10-31 21:48:41
ORD000013,CUST034,PRD005,2,2026-11-07,RETURNED,2026-11-10 11:50:56
ORD000014,CUST035,PRD015,1,2026-09-24,DELIVERED,2026-09-27 14:17:27
ORD000015,CUST017,PRD016,4,2026-10-10,SHIPPED,2026-10-13 10:22:11
ORD000016,CUST021,PRD018,4,2026-10-24,DELIVERED,2026-10-27 12:29:11
ORD000021,CUST019,PRD020,5,2026-11-12,CANCELLED,2026-11-13 19:13:36
ORD000026,CUST020,PRD004,1,2026-09-08,CANCELLED,2020-01-01 00:00:00
ORD000031,CUST008,PRD010,-1,2026-10-07,DELIVERED,2026-11-18 09:00:00
ORD000151,CUST001,PRD007,3,2026-11-13,SHIPPED,2026-11-13 19:58:00


In [0]:
%scala
val bronzeDeltaOrdersPath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze_delta/orders"

val bronzeDeltaDF = spark.read
  .format("delta")
  .load(bronzeDeltaOrdersPath)

println(s"Existing Delta orders: ${bronzeDeltaDF.count()}")
display(bronzeDeltaDF.limit(5))

order_id,customer_id,product_id,quantity,order_date,order_status,last_updated,_source_file,_ingestion_timestamp,sales_channel,discount_amount
ORD000026,CUST020,PRD004,1,2026-09-08,CANCELLED,2026-09-08 09:15:01,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z,null,null
ORD000103,CUST018,PRD012,4,2026-09-02,RETURNED,2026-09-03 15:33:29,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z,null,null
ORD000089,CUST010,PRD005,2,2026-09-20,CANCELLED,2026-09-20 17:35:45,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z,null,null
ORD000032,CUST022,PRD007,5,2026-11-14,DELIVERED,2026-11-14 22:01:34,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z,null,null
ORD000138,CUST007,PRD008,4,2026-09-23,RETURNED,2026-09-23 19:29:37,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/orders.csv,2026-11-20T17:21:45.893472Z,null,null


In [0]:
%scala
import org.apache.spark.sql.functions._

val existingOrders = bronzeDeltaDF
  .select(
    col("order_id"),
    col("last_updated").cast("timestamp").as("existing_last_updated")
  )

println(s"Existing Delta order count: ${existingOrders.count()}")

Existing Delta order count: 173
import org.apache.spark.sql.functions._
val existingOrders: org.apache.spark.sql.DataFrame = [order_id: string, existing_last_updated: timestamp]

In [0]:
%scala
val incrementalFolder =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/incremental/"

val incrementalFiles = dbutils.fs
  .ls(incrementalFolder)
  .filter(_.name.toLowerCase.endsWith(".csv"))

require(incrementalFiles.nonEmpty, "No CSV files found in incremental folder")

val incrementalDF = spark.read
  .option("header", "true")
  .option("inferSchema", "false")
  .csv(incrementalFiles.map(_.path): _*)

println(s"Files found: ${incrementalFiles.size}")
println(s"Incoming records: ${incrementalDF.count()}")

display(incrementalDF)

order_id,customer_id,product_id,quantity,order_date,order_status,last_updated
ORD000011,CUST003,PRD022,5,2026-09-26,DELIVERED,2026-09-29 18:58:11
ORD000012,CUST013,PRD017,1,2026-10-28,CANCELLED,2026-10-31 21:48:41
ORD000013,CUST034,PRD005,2,2026-11-07,RETURNED,2026-11-10 11:50:56
ORD000014,CUST035,PRD015,1,2026-09-24,DELIVERED,2026-09-27 14:17:27
ORD000015,CUST017,PRD016,4,2026-10-10,SHIPPED,2026-10-13 10:22:11
ORD000016,CUST021,PRD018,4,2026-10-24,DELIVERED,2026-10-27 12:29:11
ORD000021,CUST019,PRD020,5,2026-11-12,CANCELLED,2026-11-13 19:13:36
ORD000026,CUST020,PRD004,1,2026-09-08,CANCELLED,2020-01-01 00:00:00
ORD000031,CUST008,PRD010,-1,2026-10-07,DELIVERED,2026-11-18 09:00:00
ORD000151,CUST001,PRD007,3,2026-11-13,SHIPPED,2026-11-13 19:58:00


In [0]:
%scala
val incomingPrepared = incrementalDF
  .withColumn("quantity", col("quantity").cast("int"))
  .withColumn("incoming_last_updated",
    to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("parsed_order_date", to_date(col("order_date"), "yyyy-MM-dd"))

val joinedDF = incomingPrepared
  .join(existingOrders, Seq("order_id"), "left")

val classifiedDF = joinedDF
  .withColumn(
    "operation",
    when(
      col("order_id").isNull ||
      col("customer_id").isNull ||
      col("product_id").isNull ||
      col("quantity").isNull ||
      (col("quantity") <= 0) ||
      col("parsed_order_date").isNull ||
      col("incoming_last_updated").isNull,
      "INVALID"
    )
    .when(col("existing_last_updated").isNull, "INSERT")
    .when(
      col("incoming_last_updated") <= col("existing_last_updated"),
      "STALE"
    )
    .otherwise("UPDATE")
  )

display(
  classifiedDF.select(
    col("order_id"),
    col("quantity"),
    col("last_updated"),
    col("existing_last_updated"),
    col("operation")
  ).orderBy("order_id")
)

order_id,quantity,last_updated,existing_last_updated,operation
ORD000011,5,2026-09-29 18:58:11,2026-09-26T18:58:11Z,UPDATE
ORD000012,1,2026-10-31 21:48:41,2026-10-28T21:48:41Z,UPDATE
ORD000013,2,2026-11-10 11:50:56,2026-11-07T11:50:56Z,UPDATE
ORD000014,1,2026-09-27 14:17:27,2026-09-24T14:17:27Z,UPDATE
ORD000015,4,2026-10-13 10:22:11,2026-10-10T10:22:11Z,UPDATE
ORD000016,4,2026-10-27 12:29:11,2026-10-24T12:29:11Z,UPDATE
ORD000021,5,2026-11-13 19:13:36,2026-11-12T19:13:36Z,UPDATE
ORD000026,1,2020-01-01 00:00:00,2026-09-08T09:15:01Z,STALE
ORD000031,-1,2026-11-18 09:00:00,2026-10-07T19:21:18Z,INVALID
ORD000151,3,2026-11-13 19:58:00,null,INSERT


In [0]:
%scala
val insertDF = classifiedDF.filter(col("operation") === "INSERT")
val updateDF = classifiedDF.filter(col("operation") === "UPDATE")
val staleDF = classifiedDF.filter(col("operation") === "STALE")
val invalidDF = classifiedDF.filter(col("operation") === "INVALID")

println(s"INSERT:  ${insertDF.count()}")
println(s"UPDATE:  ${updateDF.count()}")
println(s"STALE:   ${staleDF.count()}")
println(s"INVALID: ${invalidDF.count()}")

INSERT: 8
UPDATE: 7
STALE: 1
INVALID: 1
val insertDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 9 more fields]
val updateDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 9 more fields]
val staleDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 9 more fields]
val invalidDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, customer_id: string ... 9 more fields]

In [0]:
%scala
val silverIncrementalPath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/incremental/"

insertDF.write.format("delta").mode("overwrite")
  .save(silverIncrementalPath + "inserts")

updateDF.write.format("delta").mode("overwrite")
  .save(silverIncrementalPath + "updates")

staleDF.write.format("delta").mode("overwrite")
  .save(silverIncrementalPath + "stale")

invalidDF.write.format("delta").mode("overwrite")
  .save(silverIncrementalPath + "invalid")

println("Incremental Silver datasets written successfully.")

Incremental Silver datasets written successfully.
val silverIncrementalPath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/incremental/

In [0]:
%scala
val goldFactDF = spark.table(
  "adlsdatabricks2_workspace.gold.fact_order"
)

println(s"Existing Gold orders: ${goldFactDF.count()}")

goldFactDF.printSchema()

display(goldFactDF.limit(10))

sales_key,order_id,customer_key,product_key,date_key,quantity,order_status,sales_channel,discount_amount,last_updated
1,ORD000001,21,4,20260906,1,DELIVERED,null,null,2026-09-06T20:34:17Z
2,ORD000002,29,4,20261114,5,DELIVERED,null,null,2026-11-14T17:49:59Z
3,ORD000003,30,8,20261031,1,DELIVERED,null,null,2026-10-31T17:08:38Z
4,ORD000005,12,10,20261020,1,DELIVERED,null,null,2026-10-20T20:09:33Z
5,ORD000007,19,20,20260904,3,DELIVERED,null,null,2026-09-04T08:32:30Z
6,ORD000008,27,16,20261012,5,DELIVERED,null,null,2026-10-12T22:07:31Z
7,ORD000010,5,15,20261116,4,DELIVERED,null,null,2026-11-16T11:43:45Z
8,ORD000011,3,18,20260926,5,DELIVERED,null,null,2026-09-26T18:58:11Z
9,ORD000012,12,13,20261028,1,DELIVERED,null,null,2026-10-28T21:48:41Z
10,ORD000014,31,12,20260924,1,DELIVERED,null,null,2026-09-24T14:17:27Z


In [0]:
%scala
spark.sql("""
  SHOW TABLES IN adlsdatabricks2_workspace.gold
""").show(false)

+--------+---------------------------+-----------+
database|tableName |isTemporary|
+--------+---------------------------+-----------+
gold |dim_customer |false |
gold |dim_date |false |
gold |dim_product |false |
gold |fact_order |false |
gold |vw_monthly_order_trend |false |
gold |vw_order_status_summary |false |
gold |vw_product_category_summary|false |
gold |vw_sales_channel_summary |false |
gold |vw_top_customers |false |
+--------+---------------------------+-----------+

In [0]:
%scala
val customerDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_customer"
)

val productDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_product"
)

val dateDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_date"
)

customerDim.printSchema()
productDim.printSchema()
dateDim.printSchema()

root
-- customer_key: integer (nullable = true)
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)

root
-- product_key: integer (nullable = true)
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)

root
-- date_key: integer (nullable = true)
-- full_date: date (nullable = true)
-- day: integer (nullable = true)
-- month: integer (nullable = true)
-- month_name: string (nullable = true)
-- quarter: integer (nullable = true)
-- year: integer (nullable = true)
-- day_of_week: string (nullable = true)
-- is_weekend: boolean (nullable = true)

val customerDim: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]
val productDim: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]
val dateDim: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]

In [0]:

%scala

val dimensionCheckDF = classifiedDF
  .filter(col("operation").isin("INSERT", "UPDATE"))
  .join(
    customerDim.select(
      col("customer_id"),
      col("customer_key")
    ),
    Seq("customer_id"),
    "left"
  )
  .join(
    productDim.select(
      col("product_id"),
      col("product_key")
    ),
    Seq("product_id"),
    "left"
  )
  .join(
    dateDim.select(
      col("full_date").as("parsed_order_date"),
      col("date_key")
    ),
    Seq("parsed_order_date"),
    "left"
  )

// Check dimension mapping status
dimensionCheckDF
  .withColumn(
    "dimension_status",
    when(
      col("customer_key").isNull ||
      col("product_key").isNull ||
      col("date_key").isNull,
      "MISSING_DIMENSION"
    ).otherwise("MATCHED")
  )
  .groupBy("dimension_status")
  .count()
  .show(false)

// Display records with missing dimension mappings
dimensionCheckDF
  .filter(
    col("customer_key").isNull ||
    col("product_key").isNull ||
    col("date_key").isNull
  )
  .select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("parsed_order_date"),
    col("customer_key"),
    col("product_key"),
    col("date_key")
  )
  .show(false)


+-----------------+-----+
dimension_status |count|
+-----------------+-----+
MISSING_DIMENSION|3 |
MATCHED |12 |
+-----------------+-----+

+---------+-----------+----------+-----------------+------------+-----------+--------+
order_id |customer_id|product_id|parsed_order_date|customer_key|product_key|date_key|
+---------+-----------+----------+-----------------+------------+-----------+--------+
ORD000013|CUST034 |PRD005 |2026-11-07 |NULL |4 |20261107|
ORD000015|CUST017 |PRD016 |2026-10-10 |NULL |NULL |20261010|
ORD000157|CUST017 |PRD013 |2026-11-16 |NULL |10 |20261116|
+---------+-----------+----------+-----------------+------------+-----------+--------+

val dimensionCheckDF: org.apache.spark.sql.DataFrame = [parsed_order_date: date, product_id: string ... 12 more fields]

In [0]:

%scala

println("Missing customer IDs:")
dimensionCheckDF
  .filter(col("customer_key").isNull)
  .select("customer_id")
  .distinct()
  .show(false)

println("Missing product IDs:")
dimensionCheckDF
  .filter(col("product_key").isNull)
  .select("product_id")
  .distinct()
  .show(false)


Missing customer IDs:
+-----------+
customer_id|
+-----------+
CUST017 |
CUST034 |
+-----------+

Missing product IDs:
+----------+
product_id|
+----------+
PRD016 |
+----------+

In [0]:

%scala

println("Customer IDs found in Bronze Delta:")

bronzeDeltaDF
  .filter(col("customer_id").isin("CUST017", "CUST034"))
  .select("customer_id")
  .distinct()
  .show(false)

println("Product IDs found in Bronze Delta:")

bronzeDeltaDF
  .filter(col("product_id") === "PRD016")
  .select("product_id")
  .distinct()
  .show(false)


Customer IDs found in Bronze Delta:
+-----------+
customer_id|
+-----------+
CUST017 |
CUST034 |
+-----------+

Product IDs found in Bronze Delta:
+----------+
product_id|
+----------+
PRD016 |
+----------+

In [0]:

%scala

val basePath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/"

println("BRONZE FOLDERS")
dbutils.fs.ls(basePath + "bronze/")
  .foreach(x => println(x.path))

println("\nSILVER FOLDERS")
dbutils.fs.ls(basePath + "silver/")
  .foreach(x => println(x.path))


BRONZE FOLDERS
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/orders/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/

SILVER FOLDERS
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/customer_write_test/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/customers/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/incremental/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/orders/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/products/
abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/write_test/
val basePath: String = abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/

In [0]:

%scala

println("SILVER CUSTOMERS")
dbutils.fs.ls(basePath + "silver/customers/")
  .foreach(x => println(x.name + " | " + x.path))

println("\nSILVER PRODUCTS")
dbutils.fs.ls(basePath + "silver/products/")
  .foreach(x => println(x.name + " | " + x.path))


SILVER CUSTOMERS
_delta_log/ | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/customers/_delta_log/
part-00000-da3a914b-8732-40b3-820e-60d20a98c873-c000.snappy.parquet | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/customers/part-00000-da3a914b-8732-40b3-820e-60d20a98c873-c000.snappy.parquet

SILVER PRODUCTS
_delta_log/ | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/products/_delta_log/
part-00000-26df675b-14ac-4820-8f36-b5ff3c0152ed-c000.snappy.parquet | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/silver/products/part-00000-26df675b-14ac-4820-8f36-b5ff3c0152ed-c000.snappy.parquet

In [0]:

%scala

val silverCustomerDF = spark.read
  .format("delta")
  .load(basePath + "silver/customers/")

val silverProductDF = spark.read
  .format("delta")
  .load(basePath + "silver/products/")

println("CUSTOMER SCHEMA")
silverCustomerDF.printSchema()

println("PRODUCT SCHEMA")
silverProductDF.printSchema()

println("MISSING CUSTOMER RECORDS")
silverCustomerDF
  .filter(col("customer_id").isin("CUST017", "CUST034"))
  .show(false)

println("MISSING PRODUCT RECORD")
silverProductDF
  .filter(col("product_id") === "PRD016")
  .show(false)


CUSTOMER SCHEMA
root
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

PRODUCT SCHEMA
root
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)
-- _source_file: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)

MISSING CUSTOMER RECORDS
+-----------+-------------+-----+----+-----------+------------+--------------------+
customer_id|customer_name|email|city|signup_date|_source_file|_ingestion_timestamp|
+-----------+-------------+-----+----+-----------+------------+--------------------+
+-----------+-------------+-----+----+-----------+------------+--------------------+

MISSING PRODUCT RECORD
+----------+------------+--------+----------+------+------------+--------------------+
product_id|product_name|category|unit_price|active|_source_file|_ingestion_timestamp|
+----------+------------+--------+----------+------+------------+--------------------+
+----------+------------+--------+----------+------+------------+--------------------+

val silverCustomerDF: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 5 more fields]
val silverProductDF: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 5 more fields]

In [0]:

%scala

println("BRONZE CUSTOMER FILES")
dbutils.fs.ls(basePath + "bronze/customers/")
  .foreach(x => println(x.name + " | " + x.path))

println("\nBRONZE PRODUCT FILES")
dbutils.fs.ls(basePath + "bronze/products/")
  .foreach(x => println(x.name + " | " + x.path))


BRONZE CUSTOMER FILES
customers.csv | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/customers/customers.csv

BRONZE PRODUCT FILES
products.csv | abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/nexus/batch/bronze/products/products.csv

In [0]:

%scala

val bronzeCustomerDF = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv(basePath + "bronze/customers/customers.csv")

val bronzeProductDF = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv(basePath + "bronze/products/products.csv")

println("BRONZE CUSTOMER RECORDS")
bronzeCustomerDF
  .filter(col("customer_id").isin("CUST017", "CUST034"))
  .show(false)

println("BRONZE PRODUCT RECORD")
bronzeProductDF
  .filter(col("product_id") === "PRD016")
  .show(false)


BRONZE CUSTOMER RECORDS
+-----------+-------------+------------------------------+-------+-----------+
customer_id|customer_name|email |city |signup_date|
+-----------+-------------+------------------------------+-------+-----------+
CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |
CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata|2027-06-01 |
+-----------+-------------+------------------------------+-------+-----------+

BRONZE PRODUCT RECORD
+----------+------------------+--------+----------+------+
product_id|product_name |category|unit_price|active|
+----------+------------------+--------+----------+------+
PRD016 |Green Tea 100 bags|NULL |349.0 |Y |
+----------+------------------+--------+----------+------+

val bronzeCustomerDF: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 3 more fields]
val bronzeProductDF: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 3 more fields]

In [0]:

%scala

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// Find customer records missing from Gold
val missingCustomers = bronzeCustomerDF
  .join(
    customerDim.select("customer_id"),
    Seq("customer_id"),
    "left_anti"
  )
  .select(
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("city"),
    col("signup_date")
  )

// Find product records missing from Gold
val missingProducts = bronzeProductDF
  .join(
    productDim.select("product_id"),
    Seq("product_id"),
    "left_anti"
  )
  .select(
    col("product_id"),
    col("product_name"),
    col("category"),
    col("unit_price").cast("decimal(10,2)").as("unit_price"),
    col("active")
  )

// Generate new customer surrogate keys
val maxCustomerKey = customerDim
  .agg(coalesce(max("customer_key"), lit(0)))
  .first()
  .getInt(0)

val customerWindow = Window.orderBy("customer_id")

val newCustomerDim = missingCustomers
  .withColumn(
    "customer_key",
    (row_number().over(customerWindow) + lit(maxCustomerKey)).cast("int")
  )
  .select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email",
    "city",
    "signup_date"
  )

// Generate new product surrogate keys
val maxProductKey = productDim
  .agg(coalesce(max("product_key"), lit(0)))
  .first()
  .getInt(0)

val productWindow = Window.orderBy("product_id")

val newProductDim = missingProducts
  .withColumn(
    "product_key",
    (row_number().over(productWindow) + lit(maxProductKey)).cast("int")
  )
  .select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "active"
  )

println("CUSTOMER DIMENSION RECORDS TO ADD")
newCustomerDim.show(false)

println("PRODUCT DIMENSION RECORDS TO ADD")
newProductDim.show(false)


CUSTOMER DIMENSION RECORDS TO ADD
+------------+-----------+-------------+------------------------------+---------+-----------+
customer_key|customer_id|customer_name|email |city |signup_date|
+------------+-----------+-------------+------------------------------+---------+-----------+
37 |CUST008 |Karthik Khan |invalid.email.example.in |Pune |2025-08-09 |
38 |CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |
39 |CUST026 |Vikram Khan |no@domain |Hyderabad|2023-02-03 |
40 |CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata |2027-06-01 |
+------------+-----------+-------------+------------------------------+---------+-----------+

PRODUCT DIMENSION RECORDS TO ADD
+-----------+----------+-------------------+-----------+----------+------+
product_key|product_id|product_name |category |unit_price|active|
+-----------+----------+-------------------+-----------+----------+------+
21 |PRD004 |Power Bank 20000mAh|Electronics|-1599.00 |Y |
22 |PRD009 |Air Fryer |Home |0.00 |Y |
23 |PRD011 |LED Desk Lamp |Home |899.00 |Y |
24 |PRD011 |LED Desk Lamp |Home |949.00 |Y |
25 |PRD016 |Green Tea 100 bags |NULL |349.00 |Y |
26 |PRD999 |NULL |Misc |100.00 |Y |
+-----------+----------+-------------------+-----------+----------+------+

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
val missingCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 3 more fields]
val missingProducts: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 3 more fields]
val maxCustomerKey: Int = 36
val customerWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@2b9215d1
val newCustomerDim: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]
val maxProductKey: Int = 20
val productWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@14f5ddfd
val newProductDim: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]

In [0]:

%scala

// Only customers required by the incremental orders
val requiredCustomers = bronzeCustomerDF
  .filter(col("customer_id").isin("CUST017", "CUST034"))
  .dropDuplicates(Seq("customer_id"))
  .select(
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("city"),
    col("signup_date")
  )

// Only the product required by the incremental orders
val requiredProducts = bronzeProductDF
  .filter(col("product_id") === "PRD016")
  .dropDuplicates(Seq("product_id"))
  .select(
    col("product_id"),
    col("product_name"),
    col("category"),
    col("unit_price").cast("decimal(10,2)").as("unit_price"),
    col("active")
  )

// Generate customer surrogate keys
val customerStartKey = customerDim
  .agg(coalesce(max("customer_key"), lit(0)))
  .first()
  .getInt(0)

val requiredCustomerWindow = Window.orderBy("customer_id")

val customersToAdd = requiredCustomers
  .withColumn(
    "customer_key",
    (row_number().over(requiredCustomerWindow) + lit(customerStartKey)).cast("int")
  )
  .select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email",
    "city",
    "signup_date"
  )

// Generate product surrogate key
val productStartKey = productDim
  .agg(coalesce(max("product_key"), lit(0)))
  .first()
  .getInt(0)

val productsToAdd = requiredProducts
  .withColumn(
    "product_key",
    (row_number().over(Window.orderBy("product_id")) + lit(productStartKey)).cast("int")
  )
  .select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "active"
  )

println("CUSTOMERS TO ADD")
customersToAdd.show(false)

println("PRODUCTS TO ADD")
productsToAdd.show(false)


CUSTOMERS TO ADD
+------------+-----------+-------------+------------------------------+-------+-----------+
customer_key|customer_id|customer_name|email |city |signup_date|
+------------+-----------+-------------+------------------------------+-------+-----------+
37 |CUST017 |NULL |karthik.gupta17@mailexample.in|Kochi |2024-06-09 |
38 |CUST034 |Rahul Iyer |rahul.iyer34@mailexample.in |Kolkata|2027-06-01 |
+------------+-----------+-------------+------------------------------+-------+-----------+

PRODUCTS TO ADD
+-----------+----------+------------------+--------+----------+------+
product_key|product_id|product_name |category|unit_price|active|
+-----------+----------+------------------+--------+----------+------+
21 |PRD016 |Green Tea 100 bags|NULL |349.00 |Y |
+-----------+----------+------------------+--------+----------+------+

val requiredCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 3 more fields]
val requiredProducts: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 3 more fields]
val customerStartKey: Int = 36
val requiredCustomerWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@83a732d
val customersToAdd: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]
val productStartKey: Int = 20
val productsToAdd: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]

In [0]:

%scala

import io.delta.tables.DeltaTable

// Add missing customers without duplicating existing customer IDs
DeltaTable.forName(
  "adlsdatabricks2_workspace.gold.dim_customer"
)
  .as("target")
  .merge(
    customersToAdd.as("source"),
    "target.customer_id = source.customer_id"
  )
  .whenNotMatched()
  .insertAll()
  .execute()

// Add missing product without duplicating existing product IDs
DeltaTable.forName(
  "adlsdatabricks2_workspace.gold.dim_product"
)
  .as("target")
  .merge(
    productsToAdd.as("source"),
    "target.product_id = source.product_id"
  )
  .whenNotMatched()
  .insertAll()
  .execute()

println("Gold dimension merge completed.")


Gold dimension merge completed.
import io.delta.tables.DeltaTable

In [0]:

%scala

val updatedCustomerDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_customer"
)

val updatedProductDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_product"
)

val updatedDateDim = spark.table(
  "adlsdatabricks2_workspace.gold.dim_date"
)

val revalidatedDF = classifiedDF
  .filter(col("operation").isin("INSERT", "UPDATE"))
  .join(
    updatedCustomerDim.select("customer_id", "customer_key"),
    Seq("customer_id"),
    "left"
  )
  .join(
    updatedProductDim.select("product_id", "product_key"),
    Seq("product_id"),
    "left"
  )
  .join(
    updatedDateDim.select(
      col("full_date").as("parsed_order_date"),
      col("date_key")
    ),
    Seq("parsed_order_date"),
    "left"
  )

revalidatedDF
  .withColumn(
    "dimension_status",
    when(
      col("customer_key").isNull ||
      col("product_key").isNull ||
      col("date_key").isNull,
      "MISSING_DIMENSION"
    ).otherwise("MATCHED")
  )
  .groupBy("dimension_status")
  .count()
  .show(false)

revalidatedDF
  .filter(
    col("customer_key").isNull ||
    col("product_key").isNull ||
    col("date_key").isNull
  )
  .select("order_id", "customer_id", "product_id", "parsed_order_date")
  .show(false)


+----------------+-----+
dimension_status|count|
+----------------+-----+
MATCHED |15 |
+----------------+-----+

+--------+-----------+----------+-----------------+
order_id|customer_id|product_id|parsed_order_date|
+--------+-----------+----------+-----------------+
+--------+-----------+----------+-----------------+

val updatedCustomerDim: org.apache.spark.sql.DataFrame = [customer_key: int, customer_id: string ... 4 more fields]
val updatedProductDim: org.apache.spark.sql.DataFrame = [product_key: int, product_id: string ... 4 more fields]
val updatedDateDim: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val revalidatedDF: org.apache.spark.sql.DataFrame = [parsed_order_date: date, product_id: string ... 12 more fields]

In [0]:

%scala

val goldFactDF = spark.table(
  "adlsdatabricks2_workspace.gold.fact_order"
)

println("FACT ORDER SCHEMA")
goldFactDF.printSchema()

println("CURRENT FACT TABLE COUNT")
println(goldFactDF.count())

println("MAXIMUM SALES KEY")
goldFactDF
  .agg(max("sales_key").as("max_sales_key"))
  .show(false)

println("INCOMING ORDERS ALREADY IN GOLD")
goldFactDF
  .join(
    revalidatedDF.select("order_id").distinct(),
    Seq("order_id"),
    "inner"
  )
  .select("order_id", "sales_key", "last_updated")
  .orderBy("order_id")
  .show(false)


FACT ORDER SCHEMA
root
-- sales_key: integer (nullable = true)
-- order_id: string (nullable = true)
-- customer_key: integer (nullable = true)
-- product_key: integer (nullable = true)
-- date_key: integer (nullable = true)
-- quantity: integer (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: timestamp (nullable = true)

CURRENT FACT TABLE COUNT
100
MAXIMUM SALES KEY
+-------------+
max_sales_key|
+-------------+
100 |
+-------------+

INCOMING ORDERS ALREADY IN GOLD
+---------+---------+-------------------+
order_id |sales_key|last_updated |
+---------+---------+-------------------+
ORD000011|8 |2026-09-26 18:58:11|
ORD000012|9 |2026-10-28 21:48:41|
ORD000014|10 |2026-09-24 14:17:27|
ORD000016|11 |2026-10-24 12:29:11|
ORD000021|13 |2026-11-12 19:13:36|
+---------+---------+-------------------+

val goldFactDF: org.apache.spark.sql.DataFrame = [sales_key: int, order_id: string ... 8 more fields]

In [0]:

%scala

val goldOrderKeys = goldFactDF
  .select("order_id", "sales_key")

val goldMergeCheckDF = revalidatedDF
  .join(goldOrderKeys, Seq("order_id"), "left")
  .withColumn(
    "gold_operation",
    when(col("sales_key").isNull, "INSERT")
      .otherwise("UPDATE")
  )

println("GOLD MERGE OPERATION COUNTS")

goldMergeCheckDF
  .groupBy("gold_operation")
  .count()
  .show(false)

println("ORDERS TO INSERT INTO GOLD")

goldMergeCheckDF
  .filter(col("gold_operation") === "INSERT")
  .select("order_id", "operation", "customer_key", "product_key", "date_key")
  .orderBy("order_id")
  .show(false)

println("ORDERS TO UPDATE IN GOLD")

goldMergeCheckDF
  .filter(col("gold_operation") === "UPDATE")
  .select("order_id", "operation", "sales_key")
  .orderBy("order_id")
  .show(false)


GOLD MERGE OPERATION COUNTS
+--------------+-----+
gold_operation|count|
+--------------+-----+
INSERT |10 |
UPDATE |5 |
+--------------+-----+

ORDERS TO INSERT INTO GOLD
+---------+---------+------------+-----------+--------+
order_id |operation|customer_key|product_key|date_key|
+---------+---------+------------+-----------+--------+
ORD000013|UPDATE |38 |4 |20261107|
ORD000015|UPDATE |37 |21 |20261010|
ORD000151|INSERT |1 |6 |20261113|
ORD000152|INSERT |20 |10 |20261111|
ORD000153|INSERT |22 |12 |20261111|
ORD000154|INSERT |1 |9 |20261115|
ORD000155|INSERT |1 |9 |20261117|
ORD000156|INSERT |17 |15 |20261110|
ORD000157|INSERT |37 |10 |20261116|
ORD000158|INSERT |4 |3 |20261113|
+---------+---------+------------+-----------+--------+

ORDERS TO UPDATE IN GOLD
+---------+---------+---------+
order_id |operation|sales_key|
+---------+---------+---------+
ORD000011|UPDATE |8 |
ORD000012|UPDATE |9 |
ORD000014|UPDATE |10 |
ORD000016|UPDATE |11 |
ORD000021|UPDATE |13 |
+---------+---------+---------+

val goldOrderKeys: org.apache.spark.sql.DataFrame = [order_id: string, sales_key: int]
val goldMergeCheckDF: org.apache.spark.sql.DataFrame = [order_id: string, parsed_order_date: date ... 14 more fields]

In [0]:

%scala

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// Remove duplicate incoming order IDs, retaining the latest record
val dedupWindow = Window
  .partitionBy("order_id")
  .orderBy(col("incoming_last_updated").desc)

val deduplicatedDF = goldMergeCheckDF
  .withColumn("row_num", row_number().over(dedupWindow))
  .filter(col("row_num") === 1)
  .drop("row_num")

// Get current maximum Gold sales key
val currentMaxSalesKey = goldFactDF
  .agg(max("sales_key").as("max_key"))
  .first()
  .getInt(0)

// Assign new keys only to records missing from Gold
val keyWindow = Window.orderBy(
  col("sales_key").asc_nulls_first,
  col("order_id")
)

val mergeSourceDF = deduplicatedDF
  .withColumn("key_rank", row_number().over(keyWindow))
  .withColumn(
    "final_sales_key",
    when(
      col("sales_key").isNull,
      lit(currentMaxSalesKey) + col("key_rank")
    ).otherwise(col("sales_key"))
  )
  .select(
    col("final_sales_key").cast("int").as("sales_key"),
    col("order_id"),
    col("customer_key"),
    col("product_key"),
    col("date_key"),
    col("quantity"),
    col("order_status"),
    lit(null).cast("string").as("sales_channel"),
    lit(null).cast("decimal(10,2)").as("discount_amount"),
    col("incoming_last_updated").as("last_updated")
  )

println("MERGE SOURCE COUNT")
println(mergeSourceDF.count())

println("PREPARED GOLD RECORDS")
mergeSourceDF
  .orderBy("sales_key")
  .show(20, false)

println("DUPLICATE ORDER CHECK")
mergeSourceDF
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)
  .show(false)


MERGE SOURCE COUNT
15
PREPARED GOLD RECORDS
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
sales_key|order_id |customer_key|product_key|date_key|quantity|order_status|sales_channel|discount_amount|last_updated |
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
8 |ORD000011|3 |18 |20260926|5 |DELIVERED |NULL |NULL |2026-09-29 18:58:11|
9 |ORD000012|12 |13 |20261028|1 |CANCELLED |NULL |NULL |2026-10-31 21:48:41|
10 |ORD000014|31 |12 |20260924|1 |DELIVERED |NULL |NULL |2026-09-27 14:17:27|
11 |ORD000016|19 |14 |20261024|4 |DELIVERED |NULL |NULL |2026-10-27 12:29:11|
13 |ORD000021|17 |16 |20261112|5 |CANCELLED |NULL |NULL |2026-11-13 19:13:36|
101 |ORD000013|38 |4 |20261107|2 |RETURNED |NULL |NULL |2026-11-10 11:50:56|
102 |ORD000015|37 |21 |20261010|4 |SHIPPED |NULL |NULL |2026-10-13 10:22:11|
103 |ORD000151|1 |6 |20261113|3 |SHIPPED |NULL |NULL |2026-11-13 19:58:00|
104 |ORD000152|20 |10 |20261111|4 |SHIPPED |NULL |NULL |2026-11-11 17:24:00|
105 |ORD000153|22 |12 |20261111|4 |SHIPPED |NULL |NULL |2026-11-11 17:24:00|
106 |ORD000154|1 |9 |20261115|4 |SHIPPED |NULL |NULL |2026-11-15 20:16:00|
107 |ORD000155|1 |9 |20261117|4 |SHIPPED |NULL |NULL |2026-11-17 08:38:00|
108 |ORD000156|17 |15 |20261110|1 |PLACED |NULL |NULL |2026-11-10 20:21:00|
109 |ORD000157|37 |10 |20261116|3 |SHIPPED |NULL |NULL |2026-11-16 08:13:00|
110 |ORD000158|4 |3 |20261113|2 |PLACED |NULL |NULL |2026-11-13 08:45:00|
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+

DUPLICATE ORDER CHECK
+--------+-----+
order_id|count|
+--------+-----+
+--------+-----+

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
val dedupWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@1ad6ff4d
val deduplicatedDF: org.apache.spark.sql.DataFrame = [order_id: string, parsed_order_date: date ... 14 more fields]
val currentMaxSalesKey: Int = 100
val keyWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@266d194
val mergeSourceDF: org.apache.spark.sql.DataFrame = [sales_key: int, order_id: string ... 8 more fields]

In [0]:

%scala

import io.delta.tables.DeltaTable

val goldFactTable = DeltaTable.forName(
  "adlsdatabricks2_workspace.gold.fact_order"
)

goldFactTable
  .as("target")
  .merge(
    mergeSourceDF.as("source"),
    "target.order_id = source.order_id"
  )
  .whenMatched("source.last_updated > target.last_updated")
  .updateExpr(
    Map(
      "customer_key" -> "source.customer_key",
      "product_key" -> "source.product_key",
      "date_key" -> "source.date_key",
      "quantity" -> "source.quantity",
      "order_status" -> "source.order_status",
      "last_updated" -> "source.last_updated"
    )
  )
  .whenNotMatched()
  .insertExpr(
    Map(
      "sales_key" -> "source.sales_key",
      "order_id" -> "source.order_id",
      "customer_key" -> "source.customer_key",
      "product_key" -> "source.product_key",
      "date_key" -> "source.date_key",
      "quantity" -> "source.quantity",
      "order_status" -> "source.order_status",
      "sales_channel" -> "source.sales_channel",
      "discount_amount" -> "source.discount_amount",
      "last_updated" -> "source.last_updated"
    )
  )
  .execute()

println("Incremental MERGE into Gold fact_order completed.")


Incremental MERGE into Gold fact_order completed.
import io.delta.tables.DeltaTable
val goldFactTable: io.delta.tables.DeltaTable = io.delta.tables.DeltaTable@12ca6544

In [0]:

%scala

val finalGoldFactDF = spark.table(
  "adlsdatabricks2_workspace.gold.fact_order"
)

println("FINAL GOLD FACT COUNT")
println(finalGoldFactDF.count())

println("NEWLY INSERTED ORDERS")
finalGoldFactDF
  .filter(col("order_id").isin(
    "ORD000013", "ORD000015",
    "ORD000151", "ORD000152", "ORD000153",
    "ORD000154", "ORD000155", "ORD000156",
    "ORD000157", "ORD000158"
  ))
  .select(
    "sales_key",
    "order_id",
    "customer_key",
    "product_key",
    "date_key",
    "quantity",
    "order_status",
    "last_updated"
  )
  .orderBy("sales_key")
  .show(false)

println("UPDATED EXISTING ORDERS")
finalGoldFactDF
  .filter(col("order_id").isin(
    "ORD000011", "ORD000012", "ORD000014",
    "ORD000016", "ORD000021"
  ))
  .select("order_id", "sales_key", "last_updated")
  .orderBy("sales_key")
  .show(false)


FINAL GOLD FACT COUNT
110
NEWLY INSERTED ORDERS
+---------+---------+------------+-----------+--------+--------+------------+-------------------+
sales_key|order_id |customer_key|product_key|date_key|quantity|order_status|last_updated |
+---------+---------+------------+-----------+--------+--------+------------+-------------------+
101 |ORD000013|38 |4 |20261107|2 |RETURNED |2026-11-10 11:50:56|
102 |ORD000015|37 |21 |20261010|4 |SHIPPED |2026-10-13 10:22:11|
103 |ORD000151|1 |6 |20261113|3 |SHIPPED |2026-11-13 19:58:00|
104 |ORD000152|20 |10 |20261111|4 |SHIPPED |2026-11-11 17:24:00|
105 |ORD000153|22 |12 |20261111|4 |SHIPPED |2026-11-11 17:24:00|
106 |ORD000154|1 |9 |20261115|4 |SHIPPED |2026-11-15 20:16:00|
107 |ORD000155|1 |9 |20261117|4 |SHIPPED |2026-11-17 08:38:00|
108 |ORD000156|17 |15 |20261110|1 |PLACED |2026-11-10 20:21:00|
109 |ORD000157|37 |10 |20261116|3 |SHIPPED |2026-11-16 08:13:00|
110 |ORD000158|4 |3 |20261113|2 |PLACED |2026-11-13 08:45:00|
+---------+---------+------------+-----------+--------+--------+------------+-------------------+

UPDATED EXISTING ORDERS
+---------+---------+-------------------+
order_id |sales_key|last_updated |
+---------+---------+-------------------+
ORD000011|8 |2026-09-29 18:58:11|
ORD000012|9 |2026-10-31 21:48:41|
ORD000014|10 |2026-09-27 14:17:27|
ORD000016|11 |2026-10-27 12:29:11|
ORD000021|13 |2026-11-13 19:13:36|
+---------+---------+-------------------+

val finalGoldFactDF: org.apache.spark.sql.DataFrame = [sales_key: int, order_id: string ... 8 more fields]

In [0]:

%scala

val goldSchema = "adlsdatabricks2_workspace.gold"

val dashboardViews = Seq(
  "vw_monthly_order_trend",
  "vw_order_status_summary",
  "vw_product_category_summary",
  "vw_sales_channel_summary",
  "vw_top_customers"
)

dashboardViews.foreach { viewName =>
  println(s"\n========== $viewName ==========")

  val viewDF = spark.table(s"$goldSchema.$viewName")

  println(s"Row count: ${viewDF.count()}")

  viewDF.show(20, false)
}


========== vw_monthly_order_trend ==========
Row count: 3
+----+-----+----------+------------+--------------+
year|month|month_name|total_orders|total_quantity|
+----+-----+----------+------------+--------------+
2026|9 |September |29 |86 |
2026|11 |November |47 |141 |
2026|10 |October |34 |103 |
+----+-----+----------+------------+--------------+


========== vw_order_status_summary ==========
Row count: 5
+------------+------------+
order_status|total_orders|
+------------+------------+
RETURNED |5 |
DELIVERED |52 |
CANCELLED |17 |
PLACED |20 |
SHIPPED |16 |
+------------+------------+


========== vw_product_category_summary ==========
Row count: 7
+-----------+--------------+------------+
category |total_quantity|total_orders|
+-----------+--------------+------------+
Home |61 |21 |
Sports |46 |16 |
NULL |4 |1 |
Apparel |92 |29 |
Grocery |45 |16 |
Electronics|30 |12 |
Beauty |52 |15 |
+-----------+--------------+------------+


========== vw_sales_channel_summary ==========
Row count: 4
+--------------+------------+--------------+
sales_channel |total_orders|total_quantity|
+--------------+------------+--------------+
MOBILE_APP |5 |16 |
LEGACY/UNKNOWN|100 |304 |
STORE |2 |6 |
WEB |3 |4 |
+--------------+------------+--------------+


========== vw_top_customers ==========
Row count: 10
+------------+-----------+-------------+-------+------------+--------------+
customer_key|customer_id|customer_name|city |total_orders|total_quantity|
+------------+-----------+-------------+-------+------------+--------------+
31 |CUST035 |Neha Rao |Mumbai |6 |21 |
24 |CUST027 |Rohan Gupta |Kochi |6 |19 |
1 |CUST001 |Priya Patel |Mumbai |7 |18 |
29 |CUST032 |Meera Sharma |Kochi |4 |17 |
19 |CUST021 |Meera Nair |Mumbai |7 |15 |
21 |CUST023 |Rahul Singh |Chennai|5 |15 |
10 |CUST011 |Meera Iyer |Chennai|6 |14 |
5 |CUST005 |Aarav Gupta |Mumbai |4 |14 |
27 |CUST030 |Imran Rao |Kolkata|4 |14 |
18 |CUST020 |Divya Das |Kochi |3 |14 |
+------------+-----------+-------------+-------+------------+--------------+

val goldSchema: String = adlsdatabricks2_workspace.gold
val dashboardViews: Seq[String] = List(vw_monthly_order_trend, vw_order_status_summary, vw_product_category_summary, vw_sales_channel_summary, vw_top_customers)